In [1]:
import pandas as pd
import numpy as np
import sys   
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # add project root so src is importable
import matplotlib.pyplot as plt   
import seaborn as sns 

from src.config import (
    CUSTOMERS_TRAIN, LOANS_CLEAN, TRANSACTIONS_CLEAN, TRAIN_IDS, CHURN_FEATURES,DEFAULT_FEATURES,SEGMENT_FEATURES,
) 

train_cust = pd.read_parquet(CUSTOMERS_TRAIN)  
loans = pd.read_parquet(LOANS_CLEAN)  
txns  = pd.read_parquet(TRANSACTIONS_CLEAN)   

train_ids   = pd.read_parquet(TRAIN_IDS)["customer_id"]
loans_train = loans[loans["customer_id"].isin(train_ids)].copy()
txns_train  = txns[txns["customer_id"].isin(train_ids)] .copy()

In [2]:
print(len(txns_train))
print(train_ids.nunique())
print(txns_train["customer_id"].nunique())
print(txns_train["month"].nunique())


print(train_cust["churned_12m"].value_counts(dropna=False))

122760
12000
12000
12
churned_12m
N      10854
Y        906
NaN      240
Name: count, dtype: int64


In [3]:
check=txns_train.pivot(index="customer_id",columns="month",values="txn_count").fillna(0)
print((check ==0).sum().sum())

24513


In [4]:
is_dead= (check ==0)

reversed_dead=is_dead.iloc[:,::-1]
locked = reversed_dead.cummin(axis=1)

run_length= locked.sum(axis=1)

run_length.value_counts().sort_index() 

0    9349
1    1490
2     551
3     235
4     152
5      95
6      70
7      33
8      23
9       2
Name: count, dtype: int64

In [5]:
fault=run_length > 4

pd.crosstab(train_cust.set_index("customer_id")["churned_12m"],fault)

col_0,False,True
churned_12m,,
N,10786,68
Y,753,153


In [6]:
transactions= txns_train[txns_train["month"].between("2024-07-01","2024-12-01")]

In [7]:
len(transactions)

64362

In [8]:
transactions["customer_id"].nunique()

11999

In [9]:
transactions["month"].nunique()

6

In [10]:
transactions["is_active"]=transactions["txn_count"]>0

In [11]:
transactions.columns

Index(['customer_id', 'month', 'txn_count', 'txn_value_pkr', 'is_active'], dtype='str')

In [12]:
transactions["first_months"]=transactions["txn_count"] * (transactions["month"] < "2024-10-01")
transactions["last_months"]=transactions["txn_count"] * (transactions["month"] >= "2024-10-01")

In [13]:
final_transactions=transactions.groupby("customer_id").agg(
    total_counts=("txn_count","sum"),
    total_amount=("txn_value_pkr","sum"),
    active_months= ("is_active","sum"),
    first=("first_months","sum"),
    last=("last_months","sum"),
).reset_index()
final_transactions

,customer_id,total_counts,total_amount,active_months,first,last
0,C100000,19,33630.0,4,15,4
1,C100002,21,41040.0,6,8,13
2,C100003,46,88670.0,3,46,0
3,C100006,12,20760.0,3,4,8
4,C100007,8,29560.0,4,7,1
...,...,...,...,...,...,...
11994,C114994,22,39100.0,6,9,13
11995,C114995,12,22840.0,6,8,4
11996,C114996,12,17530.0,4,6,6
11997,C114998,29,57290.0,6,11,18


In [14]:
final_transactions1=final_transactions["first"]==0
final_transactions1.sum()

np.int64(41)

In [15]:
final_transactions["difference"]= final_transactions["last"]-final_transactions["first"]
final_transactions

,customer_id,total_counts,total_amount,active_months,first,last,difference
0,C100000,19,33630.0,4,15,4,-11
1,C100002,21,41040.0,6,8,13,5
2,C100003,46,88670.0,3,46,0,-46
3,C100006,12,20760.0,3,4,8,4
4,C100007,8,29560.0,4,7,1,-6
...,...,...,...,...,...,...,...
11994,C114994,22,39100.0,6,9,13,4
11995,C114995,12,22840.0,6,8,4,-4
11996,C114996,12,17530.0,4,6,6,0
11997,C114998,29,57290.0,6,11,18,7


In [16]:
churn_features=train_cust.merge(final_transactions, on="customer_id", how="left")
churn_features

,customer_id,age,region,city,segment_true,onboarding_date,wallet_tenure_months,declared_income_band,avg_monthly_inflow_pkr,dependents,...,age_missing,income_band_missing,tenure_years,is_whale,total_counts,total_amount,active_months,first,last,difference
0,C100000,33.0,Sindh,Karachi,payroll,2024-05-14,13.0,25-50k,31700.0,1,...,0,0,1.08,0,19.0,33630.0,4.0,15.0,4.0,-11.0
1,C100002,41.0,KP,Peshawar,borrower,2024-05-08,13.0,25-50k,41000.0,1,...,0,0,1.08,0,21.0,41040.0,6.0,8.0,13.0,5.0
2,C100003,23.0,Punjab,Lahore,payroll,2024-04-10,15.0,25-50k,15800.0,1,...,0,1,1.25,0,46.0,88670.0,3.0,46.0,0.0,-46.0
3,C100006,32.0,Punjab,Multan,borrower,2023-10-03,21.0,50-100k,45500.0,2,...,0,0,1.75,0,12.0,20760.0,3.0,4.0,8.0,4.0
4,C100007,25.0,Balochistan,Quetta,saver,2024-01-17,17.0,50-100k,48900.0,1,...,0,0,1.42,0,8.0,29560.0,4.0,7.0,1.0,-6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,C114994,26.0,Punjab,Multan,saver,2023-09-27,21.0,<25k,25600.0,0,...,0,0,1.75,0,22.0,39100.0,6.0,9.0,13.0,4.0
11996,C114995,44.0,Punjab,Sargodha,saver,2024-02-19,15.0,25-50k,24500.0,2,...,0,0,1.25,0,12.0,22840.0,6.0,8.0,4.0,-4.0
11997,C114996,43.0,Punjab,Multan,payroll,2020-10-02,57.0,<25k,19700.0,2,...,0,0,4.75,0,12.0,17530.0,4.0,6.0,6.0,0.0
11998,C114998,34.0,Sindh,Sukkur,payroll,2024-05-16,13.0,25-50k,37400.0,0,...,0,0,1.08,0,29.0,57290.0,6.0,11.0,18.0,7.0


In [17]:
cols=["total_counts", "total_amount", "active_months", "first", "last", "difference"]
churn_features[cols]=churn_features[cols].fillna(0)

In [18]:
churn_features=churn_features.dropna(subset=["churned_12m"])

In [19]:
churn_features.sample(10)

,customer_id,age,region,city,segment_true,onboarding_date,wallet_tenure_months,declared_income_band,avg_monthly_inflow_pkr,dependents,...,age_missing,income_band_missing,tenure_years,is_whale,total_counts,total_amount,active_months,first,last,difference
3600,C104475,45.0,Sindh,Hyderabad,borrower,2023-07-09,24.0,25-50k,29600.0,1,...,0,0,2.00,0,23.0,57790.0,4.0,17.0,6.0,-11.0
8530,C110626,34.0,Sindh,Karachi,payroll,2020-02-28,64.0,25-50k,36900.0,2,...,0,0,5.33,0,39.0,86860.0,5.0,24.0,15.0,-9.0
8782,C110941,50.0,Islamabad,Islamabad,merchant,2023-03-07,28.0,<25k,11200.0,1,...,0,0,2.33,0,119.0,219340.0,6.0,68.0,51.0,-17.0
477,C100581,29.0,Islamabad,Islamabad,saver,2022-01-17,41.0,25-50k,33400.0,2,...,0,0,3.42,0,28.0,73080.0,6.0,13.0,15.0,2.0
10684,C113341,28.0,Punjab,Faisalabad,saver,2023-11-21,19.0,<25k,11400.0,4,...,0,0,1.58,0,29.0,68500.0,5.0,14.0,15.0,1.0
4855,C106039,44.0,Punjab,Faisalabad,borrower,2024-04-01,15.0,<25k,12500.0,2,...,0,0,1.25,0,12.0,25850.0,6.0,3.0,9.0,6.0
6857,C108556,18.0,Punjab,Multan,saver,2023-02-21,27.0,25-50k,33700.0,0,...,0,1,2.25,0,22.0,42370.0,4.0,16.0,6.0,-10.0
7675,C109568,42.0,Islamabad,Islamabad,payroll,2024-01-04,17.0,25-50k,13800.0,3,...,0,1,1.42,0,36.0,118360.0,6.0,20.0,16.0,-4.0
8575,C110677,33.0,Balochistan,Gwadar,borrower,2024-03-17,14.0,25-50k,39400.0,1,...,0,0,1.17,0,24.0,51790.0,5.0,15.0,9.0,-6.0
3978,C104947,31.0,KP,Abbottabad,saver,2023-06-23,24.0,<25k,12500.0,2,...,0,0,2.00,0,19.0,53490.0,6.0,12.0,7.0,-5.0


In [20]:
churn_features=churn_features.drop(columns=["segment_true", "tenure_years", "avg_monthly_txns", "is_whale", "city", "onboarding_date"],errors="ignore")

In [21]:
churn_features

,customer_id,age,region,wallet_tenure_months,declared_income_band,avg_monthly_inflow_pkr,dependents,smartphone_user,complaints_12m,failed_txns_12m,...,credit_score,churned_12m,age_missing,income_band_missing,total_counts,total_amount,active_months,first,last,difference
0,C100000,33.0,Sindh,13.0,25-50k,31700.0,1,1,0,1,...,442,N,0,0,19.0,33630.0,4.0,15.0,4.0,-11.0
1,C100002,41.0,KP,13.0,25-50k,41000.0,1,1,0,2,...,426,N,0,0,21.0,41040.0,6.0,8.0,13.0,5.0
2,C100003,23.0,Punjab,15.0,25-50k,15800.0,1,1,1,0,...,399,N,0,1,46.0,88670.0,3.0,46.0,0.0,-46.0
3,C100006,32.0,Punjab,21.0,50-100k,45500.0,2,1,0,3,...,489,N,0,0,12.0,20760.0,3.0,4.0,8.0,4.0
4,C100007,25.0,Balochistan,17.0,50-100k,48900.0,1,1,1,0,...,451,N,0,0,8.0,29560.0,4.0,7.0,1.0,-6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,C114994,26.0,Punjab,21.0,<25k,25600.0,0,1,0,1,...,499,N,0,0,22.0,39100.0,6.0,9.0,13.0,4.0
11996,C114995,44.0,Punjab,15.0,25-50k,24500.0,2,1,0,2,...,460,N,0,0,12.0,22840.0,6.0,8.0,4.0,-4.0
11997,C114996,43.0,Punjab,57.0,<25k,19700.0,2,1,0,2,...,525,N,0,0,12.0,17530.0,4.0,6.0,6.0,0.0
11998,C114998,34.0,Sindh,13.0,25-50k,37400.0,0,0,2,0,...,370,N,0,0,29.0,57290.0,6.0,11.0,18.0,7.0


In [22]:
churn_features.to_parquet(CHURN_FEATURES, index=False)

In [23]:
cols=["total_counts", "total_amount", "active_months", "first", "last", "difference"]

assert len(churn_features) == 11760
assert churn_features[cols].isna().sum().sum() == 0
assert churn_features["churned_12m"].isin(["Y", "N"]).all()

DEFAULT FEATURES

In [24]:
tot=loans_train["customer_id"].value_counts()
tot.value_counts()

count
1    6394
Name: count, dtype: int64

In [25]:
less=loans_train[loans_train["disbursed_date"] < "2024-09-01"]
len(less)

1168

In [26]:
less["disbursed_date"].dt.to_period("M").value_counts()

disbursed_date
2024-07    592
2024-08    576
Freq: M, Name: count, dtype: int64

In [27]:
transactions_cut= txns_train.merge(loans_train[["disbursed_date","loan_id","customer_id"]],on="customer_id",how="left")
transactions_cut

,customer_id,month,txn_count,txn_value_pkr,disbursed_date,loan_id
0,C100000,2024-07-01,6,9130.0,2025-02-23,L502380
1,C100002,2024-07-01,4,6970.0,2025-03-16,L502394
2,C100003,2024-07-01,14,21560.0,2024-12-11,L506180
3,C100007,2024-07-01,4,19590.0,NaT,NaN
4,C100008,2024-07-01,4,9510.0,2025-01-07,L504094
...,...,...,...,...,...,...
122755,C114993,2025-06-01,7,26280.0,2025-05-15,L500316
122756,C114994,2025-06-01,5,12280.0,NaT,NaN
122757,C114995,2025-06-01,0,0.0,2024-11-10,L507630
122758,C114998,2025-06-01,5,11310.0,NaT,NaN


In [28]:
transactions_cut.isna().sum()

customer_id           0
month                 0
txn_count             0
txn_value_pkr         0
disbursed_date    57305
loan_id           57305
dtype: int64

In [29]:
filter_months=transactions_cut[transactions_cut["month"] < transactions_cut["disbursed_date"]]
filter_months.sample(10)

,customer_id,month,txn_count,txn_value_pkr,disbursed_date,loan_id
10194,C113563,2024-07-01,3,4140.0,2024-12-12,L506275
72470,C112023,2025-01-01,12,11790.0,2025-04-04,L506978
46268,C103688,2024-11-01,3,3950.0,2025-03-26,L505499
30594,C111524,2024-09-01,3,6840.0,2025-01-24,L500850
67431,C104503,2025-01-01,9,13870.0,2025-05-01,L500460
22677,C100582,2024-09-01,17,99150.0,2024-12-24,L504518
22876,C100862,2024-09-01,2,5210.0,2025-04-09,L502054
19703,C111484,2024-08-01,4,2420.0,2024-09-02,L505143
43945,C100363,2024-11-01,2,3210.0,2025-02-25,L505176
83876,C114249,2025-02-01,0,0.0,2025-05-08,L507355


In [30]:
len(filter_months)

33732

In [31]:
print(filter_months["loan_id"].nunique())

6328


In [32]:
filter_months["is_active"]=filter_months["txn_count"]>0
filter_months

,customer_id,month,txn_count,txn_value_pkr,disbursed_date,loan_id,is_active
0,C100000,2024-07-01,6,9130.0,2025-02-23,L502380,True
1,C100002,2024-07-01,4,6970.0,2025-03-16,L502394,True
2,C100003,2024-07-01,14,21560.0,2024-12-11,L506180,True
4,C100008,2024-07-01,4,9510.0,2025-01-07,L504094,True
6,C100011,2024-07-01,22,20870.0,2024-10-20,L507343,True
...,...,...,...,...,...,...,...
113178,C114826,2025-05-01,11,15300.0,2025-05-11,L504002,True
113185,C114834,2025-05-01,8,29250.0,2025-05-16,L500665,True
113217,C114882,2025-05-01,5,9520.0,2025-05-05,L506890,True
113263,C114952,2025-05-01,2,5050.0,2025-05-13,L502301,True


In [33]:
final_default=filter_months.groupby("loan_id").agg(
    total_txns=("txn_count","sum"),
    total_value=("txn_value_pkr","sum"),
    active_months=("is_active","sum"),
).reset_index()

In [34]:
gap= (loans_train["disbursed_date"].dt.to_period("M") - pd.Period("2024-07", freq="M")).apply(lambda x: x.n)+1
gap

3        2
4        4
5        8
6        7
7        8
        ..
7994     2
7995    11
7996     7
7997     6
7998    10
Name: disbursed_date, Length: 6394, dtype: int64

In [35]:
gap.min()

np.int64(1)

In [36]:
loans_train["months_available"]=gap
loans_train

,loan_id,customer_id,disbursed_date,purpose,amount_pkr,term_months,interest_rate_pct,inflow_to_loan_ratio,defaulted,amount_suspect,months_available
3,L500003,C112525,2024-08-15,nano_loan,12857.0,3,33.3,0.48,0,False,2
4,L500004,C104772,2024-10-22,nano_loan,22631.0,3,29.4,0.75,0,False,4
5,L500005,C111618,2025-02-25,merchant_advance,158361.0,1,33.6,6.09,1,False,8
6,L500006,C102207,2025-01-13,nano_loan,21081.0,6,31.4,1.33,0,False,7
7,L500007,C100980,2025-02-12,nano_loan,11358.0,3,34.3,0.26,1,False,8
...,...,...,...,...,...,...,...,...,...,...,...
7994,L507994,C108628,2024-08-05,nano_loan,17440.0,1,30.4,0.45,0,False,2
7995,L507995,C105831,2025-05-26,device_finance,82169.0,1,29.9,2.79,0,False,11
7996,L507996,C110319,2025-01-12,merchant_advance,134329.0,1,32.9,5.46,0,False,7
7997,L507997,C103624,2024-12-31,nano_loan,24820.0,3,28.3,0.91,0,False,6


In [37]:
feature_default=loans_train.merge(final_default,on="loan_id",how="left")
feature_default

,loan_id,customer_id,disbursed_date,purpose,amount_pkr,term_months,interest_rate_pct,inflow_to_loan_ratio,defaulted,amount_suspect,months_available,total_txns,total_value,active_months
0,L500003,C112525,2024-08-15,nano_loan,12857.0,3,33.3,0.48,0,False,2,35.0,63190.0,2.0
1,L500004,C104772,2024-10-22,nano_loan,22631.0,3,29.4,0.75,0,False,4,92.0,170330.0,4.0
2,L500005,C111618,2025-02-25,merchant_advance,158361.0,1,33.6,6.09,1,False,8,103.0,200000.0,8.0
3,L500006,C102207,2025-01-13,nano_loan,21081.0,6,31.4,1.33,0,False,7,28.0,67180.0,6.0
4,L500007,C100980,2025-02-12,nano_loan,11358.0,3,34.3,0.26,1,False,8,33.0,59620.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6389,L507994,C108628,2024-08-05,nano_loan,17440.0,1,30.4,0.45,0,False,2,5.0,9460.0,1.0
6390,L507995,C105831,2025-05-26,device_finance,82169.0,1,29.9,2.79,0,False,11,51.0,149420.0,10.0
6391,L507996,C110319,2025-01-12,merchant_advance,134329.0,1,32.9,5.46,0,False,7,27.0,63020.0,6.0
6392,L507997,C103624,2024-12-31,nano_loan,24820.0,3,28.3,0.91,0,False,6,7.0,19990.0,5.0


In [38]:
cols=["total_txns", "total_value", "active_months"]
feature_default[cols]=feature_default[cols].fillna(0)

In [39]:
assert (feature_default["active_months"] <= feature_default["months_available"]).all()

In [40]:
feature_default["months_available"].isnull().sum()

np.int64(0)

In [41]:
feature_default["average_txns_per_mon"]= feature_default["total_txns"] / feature_default["months_available"]
feature_default["average_value_per_mon"]= feature_default["total_value"] / feature_default["months_available"]
feature_default["active_ratio"]= feature_default["active_months"] / feature_default["months_available"]

In [42]:
feature_default.sample(10)

,loan_id,customer_id,disbursed_date,purpose,amount_pkr,term_months,interest_rate_pct,inflow_to_loan_ratio,defaulted,amount_suspect,months_available,total_txns,total_value,active_months,average_txns_per_mon,average_value_per_mon,active_ratio
1049,L501332,C100352,2025-03-30,nano_loan,8442.0,3,33.2,0.09,0,False,9,231.0,572170.0,9.0,25.666667,63574.444444,1.000000
2880,L503617,C107403,2025-02-16,device_finance,109067.0,1,34.0,7.04,1,False,8,23.0,68360.0,6.0,2.875000,8545.000000,0.750000
3222,L504034,C100963,2025-04-17,device_finance,83948.0,3,21.8,2.10,0,False,10,171.0,467400.0,10.0,17.100000,46740.000000,1.000000
809,L501030,C109527,2024-07-21,nano_loan,3942.0,1,32.9,0.09,0,False,1,4.0,9660.0,1.0,4.000000,9660.000000,1.000000
327,L500411,C108261,2025-03-09,nano_loan,22404.0,3,26.3,0.47,0,False,9,48.0,110480.0,8.0,5.333333,12275.555556,0.888889
480,L500601,C103066,2025-01-23,emergency,6789.0,6,32.5,0.27,0,False,7,120.0,278190.0,7.0,17.142857,39741.428571,1.000000
2607,L503267,C108372,2025-03-27,merchant_advance,128992.0,3,26.8,2.66,0,False,9,52.0,143650.0,8.0,5.777778,15961.111111,0.888889
2314,L502905,C104043,2024-08-30,emergency,31853.0,3,34.1,0.89,1,False,2,4.0,9350.0,2.0,2.000000,4675.000000,1.000000
1769,L502239,C105005,2025-04-05,nano_loan,7773.0,6,30.2,0.80,0,False,10,13.0,22790.0,5.0,1.300000,2279.000000,0.500000
2358,L502959,C109485,2025-05-08,merchant_advance,343768.0,3,34.8,22.92,1,False,11,14.0,27770.0,5.0,1.272727,2524.545455,0.454545


In [43]:
feature_default["months_available"].min()

np.int64(1)

In [44]:
final_feature_default=feature_default.merge(train_cust,on="customer_id",how="left")
final_feature_default 

,loan_id,customer_id,disbursed_date,purpose,amount_pkr,term_months,interest_rate_pct,inflow_to_loan_ratio,defaulted,amount_suspect,...,failed_txns_12m,has_savings,savings_balance_pkr,has_insurance,credit_score,churned_12m,age_missing,income_band_missing,tenure_years,is_whale
0,L500003,C112525,2024-08-15,nano_loan,12857.0,3,33.3,0.48,0,False,...,1,0,0.0,0,444,N,0,0,1.75,0
1,L500004,C104772,2024-10-22,nano_loan,22631.0,3,29.4,0.75,0,False,...,2,0,0.0,0,465,N,0,0,1.75,0
2,L500005,C111618,2025-02-25,merchant_advance,158361.0,1,33.6,6.09,1,False,...,0,0,0.0,0,421,N,0,1,1.58,0
3,L500006,C102207,2025-01-13,nano_loan,21081.0,6,31.4,1.33,0,False,...,1,0,0.0,1,471,N,0,0,3.17,0
4,L500007,C100980,2025-02-12,nano_loan,11358.0,3,34.3,0.26,1,False,...,4,0,0.0,1,416,N,0,0,1.17,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6389,L507994,C108628,2024-08-05,nano_loan,17440.0,1,30.4,0.45,0,False,...,2,1,63900.0,0,483,N,0,0,2.92,0
6390,L507995,C105831,2025-05-26,device_finance,82169.0,1,29.9,2.79,0,False,...,1,0,0.0,0,426,N,0,0,2.17,0
6391,L507996,C110319,2025-01-12,merchant_advance,134329.0,1,32.9,5.46,0,False,...,2,0,0.0,0,428,N,0,0,1.83,0
6392,L507997,C103624,2024-12-31,nano_loan,24820.0,3,28.3,0.91,0,False,...,2,1,27000.0,0,511,N,0,0,3.75,0


In [45]:
cols1=["segment_true", "tenure_years", "avg_monthly_txns", "is_whale", "city", "onboarding_date", "amount_suspect", "churned_12m", "interest_rate_pct"]

default_features=final_feature_default.drop(columns=cols1)
default_features

,loan_id,customer_id,disbursed_date,purpose,amount_pkr,term_months,inflow_to_loan_ratio,defaulted,months_available,total_txns,...,dependents,smartphone_user,complaints_12m,failed_txns_12m,has_savings,savings_balance_pkr,has_insurance,credit_score,age_missing,income_band_missing
0,L500003,C112525,2024-08-15,nano_loan,12857.0,3,0.48,0,2,35.0,...,0,1,1,1,0,0.0,0,444,0,0
1,L500004,C104772,2024-10-22,nano_loan,22631.0,3,0.75,0,4,92.0,...,2,1,0,2,0,0.0,0,465,0,0
2,L500005,C111618,2025-02-25,merchant_advance,158361.0,1,6.09,1,8,103.0,...,0,1,2,0,0,0.0,0,421,0,1
3,L500006,C102207,2025-01-13,nano_loan,21081.0,6,1.33,0,7,28.0,...,1,1,1,1,0,0.0,1,471,0,0
4,L500007,C100980,2025-02-12,nano_loan,11358.0,3,0.26,1,8,33.0,...,1,1,0,4,0,0.0,1,416,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6389,L507994,C108628,2024-08-05,nano_loan,17440.0,1,0.45,0,2,5.0,...,1,1,1,2,1,63900.0,0,483,0,0
6390,L507995,C105831,2025-05-26,device_finance,82169.0,1,2.79,0,11,51.0,...,0,1,0,1,0,0.0,0,426,0,0
6391,L507996,C110319,2025-01-12,merchant_advance,134329.0,1,5.46,0,7,27.0,...,2,1,1,2,0,0.0,0,428,0,0
6392,L507997,C103624,2024-12-31,nano_loan,24820.0,3,0.91,0,6,7.0,...,2,1,0,2,1,27000.0,0,511,0,0


In [46]:
default_features.to_parquet(DEFAULT_FEATURES, index=False)

In [47]:
assert "segment_true" not in churn_features.columns

K-MEANS SEGMENTATION

In [48]:
txns_train["is_active"]=txns_train["txn_count"]>0
txns_train

,customer_id,month,txn_count,txn_value_pkr,is_active
0,C100000,2024-07-01,6,9130.0,True
2,C100002,2024-07-01,4,6970.0,True
3,C100003,2024-07-01,14,21560.0,True
6,C100007,2024-07-01,4,19590.0,True
7,C100008,2024-07-01,4,9510.0,True
...,...,...,...,...,...
153338,C114993,2025-06-01,7,26280.0,True
153339,C114994,2025-06-01,5,12280.0,True
153340,C114995,2025-06-01,0,0.0,False
153342,C114998,2025-06-01,5,11310.0,True


In [49]:
final_segment=txns_train.groupby("customer_id").agg(
    total_txns=("txn_count","sum"),
    total_value=("txn_value_pkr","sum"),
    active_months=("is_active","sum"),
).reset_index()

In [50]:
final_segment.sample(10)

,customer_id,total_txns,total_value,active_months
487,C100593,60,134080.0,12
9386,C111697,72,188100.0,9
1710,C102132,126,243990.0,12
5924,C107360,21,103350.0,10
7075,C108819,47,127560.0,11
6392,C107965,59,152420.0,12
9286,C111574,74,195190.0,12
8778,C110936,95,223540.0,5
3666,C104549,65,131410.0,11
3215,C103998,41,129430.0,12


In [51]:
segment_features_final=train_cust.merge(final_segment,on="customer_id",how="left")
segment_features_final

,customer_id,age,region,city,segment_true,onboarding_date,wallet_tenure_months,declared_income_band,avg_monthly_inflow_pkr,dependents,...,has_insurance,credit_score,churned_12m,age_missing,income_band_missing,tenure_years,is_whale,total_txns,total_value,active_months
0,C100000,33.0,Sindh,Karachi,payroll,2024-05-14,13.0,25-50k,31700.0,1,...,0,442,N,0,0,1.08,0,34,60460.0,7
1,C100002,41.0,KP,Peshawar,borrower,2024-05-08,13.0,25-50k,41000.0,1,...,1,426,N,0,0,1.08,0,33,78460.0,11
2,C100003,23.0,Punjab,Lahore,payroll,2024-04-10,15.0,25-50k,15800.0,1,...,0,399,N,0,1,1.25,0,80,159680.0,6
3,C100006,32.0,Punjab,Multan,borrower,2023-10-03,21.0,50-100k,45500.0,2,...,0,489,N,0,0,1.75,0,49,103940.0,9
4,C100007,25.0,Balochistan,Quetta,saver,2024-01-17,17.0,50-100k,48900.0,1,...,0,451,N,0,0,1.42,0,29,116010.0,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,C114994,26.0,Punjab,Multan,saver,2023-09-27,21.0,<25k,25600.0,0,...,1,499,N,0,0,1.75,0,40,77910.0,11
11996,C114995,44.0,Punjab,Sargodha,saver,2024-02-19,15.0,25-50k,24500.0,2,...,1,460,N,0,0,1.25,0,23,45830.0,11
11997,C114996,43.0,Punjab,Multan,payroll,2020-10-02,57.0,<25k,19700.0,2,...,0,525,N,0,0,4.75,0,26,65280.0,7
11998,C114998,34.0,Sindh,Sukkur,payroll,2024-05-16,13.0,25-50k,37400.0,0,...,0,370,N,0,0,1.08,0,58,114010.0,12


In [52]:
cols2=["segment_true", "churned_12m", "tenure_years", "avg_monthly_txns", "is_whale", "city", "onboarding_date", "age_missing", "income_band_missing"]

segment_features= segment_features_final.drop(columns=cols2)
segment_features

,customer_id,age,region,wallet_tenure_months,declared_income_band,avg_monthly_inflow_pkr,dependents,smartphone_user,complaints_12m,failed_txns_12m,has_savings,savings_balance_pkr,has_insurance,credit_score,total_txns,total_value,active_months
0,C100000,33.0,Sindh,13.0,25-50k,31700.0,1,1,0,1,0,0.0,0,442,34,60460.0,7
1,C100002,41.0,KP,13.0,25-50k,41000.0,1,1,0,2,0,0.0,1,426,33,78460.0,11
2,C100003,23.0,Punjab,15.0,25-50k,15800.0,1,1,1,0,1,10000.0,0,399,80,159680.0,6
3,C100006,32.0,Punjab,21.0,50-100k,45500.0,2,1,0,3,1,13000.0,0,489,49,103940.0,9
4,C100007,25.0,Balochistan,17.0,50-100k,48900.0,1,1,1,0,1,19900.0,0,451,29,116010.0,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11995,C114994,26.0,Punjab,21.0,<25k,25600.0,0,1,0,1,1,69700.0,1,499,40,77910.0,11
11996,C114995,44.0,Punjab,15.0,25-50k,24500.0,2,1,0,2,1,12500.0,1,460,23,45830.0,11
11997,C114996,43.0,Punjab,57.0,<25k,19700.0,2,1,0,2,1,23600.0,0,525,26,65280.0,7
11998,C114998,34.0,Sindh,13.0,25-50k,37400.0,0,0,2,0,0,0.0,0,370,58,114010.0,12


In [53]:
segment_features.to_parquet(SEGMENT_FEATURES, index=False) 

In [55]:
# 1. confirm the saved parquet is clean
print("segment_true" in pd.read_parquet(CHURN_FEATURES).columns)   # expect False

# 2. does active_months just count rows, or real activity?
rows_in_window = transactions.groupby("customer_id").size()
merged = churn_features.set_index("customer_id")["active_months"]
print((merged == rows_in_window.reindex(merged.index)).all())      # True = counts rows only

False
False
